# Exploratory Data Analysis: Amazon-Google

Understand the dataset before building features. Key questions:
- What do the tables look like? How many records, what fields, how much missing data?
- What do true match pairs vs non-match pairs look like?
- What patterns will features need to exploit?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_a = pd.read_csv("../data/amazon-google/tableA.csv")
df_b = pd.read_csv("../data/amazon-google/tableB.csv")
train = pd.read_csv("../data/amazon-google/train.csv")
valid = pd.read_csv("../data/amazon-google/valid.csv")
test = pd.read_csv("../data/amazon-google/test.csv")

print(f"Table A: {len(df_a)} records, columns: {list(df_a.columns)}")
print(f"Table B: {len(df_b)} records, columns: {list(df_b.columns)}")
print(f"Splits: train={len(train)}, valid={len(valid)}, test={len(test)}")
print(f"\nLabel distribution:")
for name, split in [("train", train), ("valid", valid), ("test", test)]:
    pos = split.label.sum()
    print(f"  {name}: {pos} matches / {len(split)} pairs ({pos/len(split)*100:.1f}%)")

## Missing Data

In [ ]:
print("Table A missing values:")
print(df_a.isnull().sum())
print(f"\nTable B missing values:")
print(df_b.isnull().sum())
print(f"\nKey: Table B manufacturer is missing {df_b.manufacturer.isnull().sum()}/{len(df_b)} "
      f"({df_b.manufacturer.isnull().mean()*100:.0f}%) -- features must handle NaN gracefully.")

## Sample Match and Non-Match Pairs

Looking at what the model needs to distinguish.

In [ ]:
def show_pairs(split, label_val, n=5):
    subset = split[split.label == label_val].head(n)
    for _, row in subset.iterrows():
        ra = df_a[df_a.id == row.ltable_id].iloc[0]
        rb = df_b[df_b.id == row.rtable_id].iloc[0]
        print(f"  A: {ra.title}")
        print(f"     mfr={ra.manufacturer}, price={ra.price}")
        print(f"  B: {rb.title}")
        print(f"     mfr={rb.manufacturer}, price={rb.price}")
        print()

print("=== TRUE MATCHES ===")
show_pairs(train, 1)
print("=== NON-MATCHES ===")
show_pairs(train, 0)

## Price Distribution

Price is a strong structured signal. How does it distribute across tables?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_a.price.dropna().clip(upper=500).hist(bins=50, ax=axes[0], alpha=0.7)
axes[0].set_title(f"Table A prices (n={df_a.price.notna().sum()}, {df_a.price.isna().sum()} missing)")
axes[0].set_xlabel("Price ($)")
df_b.price.dropna().clip(upper=500).hist(bins=50, ax=axes[1], alpha=0.7, color="orange")
axes[1].set_title(f"Table B prices (n={df_b.price.notna().sum()}, {df_b.price.isna().sum()} missing)")
axes[1].set_xlabel("Price ($)")
plt.tight_layout()
plt.show()

print(f"Table A price stats:\n{df_a.price.describe()}")
print(f"\nTable B price stats:\n{df_b.price.describe()}")

## Title Length Distribution

Longer titles in Table B suggest more verbose product descriptions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_a.title.str.len().hist(bins=50, ax=ax, alpha=0.6, label=f"Table A (median={df_a.title.str.len().median():.0f})")
df_b.title.str.len().hist(bins=50, ax=ax, alpha=0.6, label=f"Table B (median={df_b.title.str.len().median():.0f})")
ax.set_xlabel("Title length (chars)")
ax.set_title("Title Length Distribution")
ax.legend()
plt.tight_layout()
plt.show()